In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import corner
import ultranest
from utils.parse_data import *

In [7]:
df = read_file("data/mass.txt")
df = extract_relevant_data(df)

print(df.head())

     A    Z    N  EL  binding_energy_keV  binding_energy_unc
0  1.0  0.0  1.0   n             0.00000             0.00000
1  1.0  1.0  0.0   H             0.00000             0.00000
2  2.0  1.0  1.0   H          1112.28310             0.00020
3  3.0  1.0  2.0   H          2827.26540             0.00030
4  3.0  2.0  1.0  He          2572.68044             0.00015


**Likelihood and systematic uncertainty**

We use a Gaussian likelihood with an extra systematic term: $$\sigma_{\mathrm{sys}}$$ so that:

$$\sigma_i^2 = \sigma_{i,\mathrm{exp}}^2 + \sigma_{\mathrm{sys}}^2.$$

The log-likelihood becomes:

$$\log L = -\tfrac{1}{2}\sum_i\left[\frac{(B_i^{\mathrm{obs}}-B_i^{\mathrm{mod}})^2}{\sigma_i^2} + \ln(2\pi\sigma_i^2)\right]$$

In [8]:
# Bethe-Weizsäcker model WITHOUT pairing term, with prior and likelihood (uses same sigma_sys treatment)
def bethe_weizsacker_per_nucleon_no_pairing(A, Z, a_v, a_s, a_c, a_sym):
    A = np.array(A, dtype=float)
    Z = np.array(Z, dtype=float)
    B = a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A
    return B/A  # Return binding energy per nucleon

# Prior transform for no-pairing model: 5 parameters (a_v,a_s,a_c,a_sym,sigma_sys)
def prior_transform_nopair(u):
    u = np.asarray(u)
    if u.ndim == 1:
        a_v = 5.0 + u[0]*35.0      # 5-40 MeV
        a_s = 0.0 + u[1]*50.0      # 0-50 MeV
        a_c = 0.0 + u[2]*1.5       # 0-1.5 MeV
        a_sym = 0.0 + u[3]*60.0    # 0-60 MeV
        log10_sigma = -3.0 + u[4]*7.0  # sigma_sys from 1e-3 to 10 (same units as B)
        sigma_sys = 10**log10_sigma
        return [a_v, a_s, a_c, a_sym, sigma_sys]
    a_v = 5.0 + u[:, 0]*35.0      # 5-40 MeV
    a_s = 0.0 + u[:, 1]*50.0      # 0-50 MeV
    a_c = 0.0 + u[:, 2]*1.5       # 0-1.5 MeV
    a_sym = 0.0 + u[:, 3]*60.0    # 0-60 MeV
    log10_sigma = -3.0 + u[:, 4]*7.0  # sigma_sys from 1e-3 to 10 (same units as B)
    sigma_sys = 10**log10_sigma
    return np.column_stack([a_v, a_s, a_c, a_sym, sigma_sys])

def log_likelihood_nopair(params, A, Z, B_obs, B_err):
    params = np.atleast_2d(params)
    a_v = params[:, 0][:, None]
    a_s = params[:, 1][:, None]
    a_c = params[:, 2][:, None]
    a_sym = params[:, 3][:, None]
    sigma_sys = params[:, 4][:, None]

    A = A[None, :]
    Z = Z[None, :]
    B_obs = B_obs[None, :]
    B_err = B_err[None, :]

    B_model = (a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A) / A
    sigma2 = B_err**2 + sigma_sys**2
    ll = -0.5*np.sum((B_obs - B_model)**2/sigma2 + np.log(2*np.pi*sigma2), axis=1)
    return ll[0] if params.shape[0] == 1 else ll

print('Defined no-pairing model, prior_transform_nopair and log_likelihood_nopair')

Defined no-pairing model, prior_transform_nopair and log_likelihood_nopair


In [9]:
# Bethe-Weizsäcker model WITH pairing term, with prior and likelihood (uses same sigma_sys treatment)
def bethe_weizsacker_per_nucleon_with_pairing(A, Z, a_v, a_s, a_c, a_sym, a_p):
    A = np.array(A, dtype=float)
    Z = np.array(Z, dtype=float)
    pairing_term = np.zeros_like(A)
    even_Z = (Z % 2 == 0)
    even_N = ((A - Z) % 2 == 0)
    pairing_term[even_Z & even_N] = a_p / A[even_Z & even_N]**0.5
    pairing_term[~even_Z & ~even_N] = -a_p / A[~even_Z & ~even_N]**0.5
    B = (a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A + pairing_term)
    return B/A # Return binding energy per nucleon

def prior_transform_with_pairing(u):
    u = np.asarray(u)
    if u.ndim == 1:
        a_v = 5.0 + u[0]*35.0      # 5-40 MeV
        a_s = 0.0 + u[1]*50.0      # 0-50 MeV
        a_c = 0.0 + u[2]*1.5       # 0-1.5 MeV
        a_sym = 0.0 + u[3]*60.0    # 0-60 MeV
        a_p = 0.0 + u[4]*30.0      # 0-30 MeV (pairing term)
        log10_sigma = -3.0 + u[5]*4.0  # sigma_sys from 1e-3 to 10 (same units as B)
        sigma_sys = 10**log10_sigma
        return [a_v, a_s, a_c, a_sym, a_p, sigma_sys]
    a_v = 5.0 + u[:, 0]*35.0      # 5-40 MeV
    a_s = 0.0 + u[:, 1]*50.0      # 0-50 MeV
    a_c = 0.0 + u[:, 2]*1.5       # 0-1.5 MeV
    a_sym = 0.0 + u[:, 3]*60.0    # 0-60 MeV
    a_p = 0.0 + u[:, 4]*30.0      # 0-30 MeV (pairing term)
    log10_sigma = -3.0 + u[:, 5]*4.0  # sigma_sys from 1e-3 to 10 (same units as B)
    sigma_sys = 10**log10_sigma
    return np.column_stack([a_v, a_s, a_c, a_sym, a_p, sigma_sys])

def log_likelihood_with_pairing(params, A, Z, B_obs, B_err, pair_factor):
    params = np.atleast_2d(params)
    a_v = params[:, 0][:, None]
    a_s = params[:, 1][:, None]
    a_c = params[:, 2][:, None]
    a_sym = params[:, 3][:, None]
    a_p = params[:, 4][:, None]
    sigma_sys = params[:, 5][:, None]

    A = A[None, :]
    Z = Z[None, :]
    B_obs = B_obs[None, :]
    B_err = B_err[None, :]

    pairing_term = a_p * pair_factor[None, :]
    B_model = (a_v*A - a_s*A**(2.0/3.0) - a_c*Z*(Z-1)/A**(1.0/3.0) - a_sym*(A-2*Z)**2/A + pairing_term) / A
    sigma2 = B_err**2 + sigma_sys**2
    ll = -0.5*np.sum((B_obs - B_model)**2/sigma2 + np.log(2*np.pi*sigma2), axis=1)
    return ll[0] if params.shape[0] == 1 else ll

In [12]:
# Dati osservati
A_data = df['A'].to_numpy().astype(float)
Z_data = df['Z'].to_numpy().astype(float)
B_obs_data = df['binding_energy_keV'].to_numpy().astype(float)
B_err_data = df['binding_energy_unc'].to_numpy().astype(float)

pair_factor = np.zeros_like(A_data, dtype=float)
even_Z = (Z_data % 2 == 0)
even_N = ((A_data - Z_data) % 2 == 0)
pair_factor[even_Z & even_N] = 1.0 / np.sqrt(A_data[even_Z & even_N])
pair_factor[~even_Z & ~even_N] = -1.0 / np.sqrt(A_data[~even_Z & ~even_N])

def loglike_nopair(theta):
    return log_likelihood_nopair(theta, A_data, Z_data, B_obs_data, B_err_data)

def loglike_with_pairing(theta):
    return log_likelihood_with_pairing(theta, A_data, Z_data, B_obs_data, B_err_data, pair_factor)

sampler_nopair = ultranest.ReactiveNestedSampler(
    ['a_v', 'a_s', 'a_c', 'a_sym', 'sigma_sys'],
    loglike_nopair,
    prior_transform_nopair,
    log_dir='ultranest_nopair',
    vectorized=True
)
result_nopair = sampler_nopair.run(min_num_live_points=300, dlogz=0.05)

print("No-pairing model:")
print("  logZ =", result_nopair['logz'], "±", result_nopair['logzerr'])
print("Parameters (no-pairing):", result_nopair['samples'].mean(axis=0))

Creating directory for new run ultranest_nopair/run8
[ultranest] To achieve the desired logz accuracy, min_num_live_points was increased to 633
[ultranest] Sampling 633 live points from prior ...


[ultranest] Explored until L=-3e+04  -26547.95 [-26548.6465..-26548.6463]*| it/evals=9324/24219 eff=39.5319% N=633 3  
[ultranest] Likelihood function evaluations: 24238
[ultranest] Writing samples and results to disk ...
[ultranest] Writing samples and results to disk ... done
[ultranest]   logZ = -2.656e+04 +- 0.07807
[ultranest] Effective samples strategy satisfied (ESS = 3782.5, need >400)
[ultranest] Posterior uncertainty strategy is satisfied (KL: 0.46+-0.04 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 1996 minimum live points (dlogz from 0.06 to 0.25, need <0.05)
[ultranest]   logZ error budget: single: 0.11 bs:0.08 tail:0.01 total:0.08 required:<0.05
[ultranest] Widening roots to 1996 live points (have 633 already) ...
[ultranest] Sampling 1363 live points from prior ...
[ultranest] Explored until L=-3e+04  -26547.95 [-26548.6570..-26548.6569]*| it/evals=29706/75841 eff=40.6429% N=1996 6   
[ultranest] Likelihood function evaluations: 75855
[ultranest] W

KeyboardInterrupt: 

In [ ]:
sampler_pairing = ultranest.ReactiveNestedSampler(
    ['a_v', 'a_s', 'a_c', 'a_sym', 'a_p', 'sigma_sys'],
    loglike_with_pairing,
    prior_transform_with_pairing,
    log_dir='ultranest_pairing',
    vectorized=True
)
result_pairing = sampler_pairing.run(min_num_live_points=300, dlogz=0.5)


print("With-pairing model:")
print("  logZ =", result_pairing['logz'], "±", result_pairing['logzerr'])
print("Bayes factor (pairing / no-pairing) =", np.exp(result_pairing['logz'] - result_nopair['logz']))
print("Parameters (no-pairing):", result_nopair['samples'].mean(axis=0))
print("Parameters (with-pairing):", result_pairing['samples'].mean(axis=0))

Creating directory for new run ultranest_nopair/run6
[ultranest] Sampling 300 live points from prior ...


[ultranest] Explored until L=-3e+04  -26548.08 [-26548.6782..-26548.6774]*| it/evals=4440/12430 eff=36.6035% N=300   
[ultranest] Likelihood function evaluations: 12471
[ultranest] Writing samples and results to disk ...
[ultranest] Writing samples and results to disk ... done
[ultranest]   logZ = -2.656e+04 +- 0.1246
[ultranest] Effective samples strategy satisfied (ESS = 1874.4, need >400)
[ultranest] Posterior uncertainty strategy is satisfied (KL: 0.45+-0.06 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy is satisfied (dlogz=0.12, need <0.5)
[ultranest]   logZ error budget: single: 0.16 bs:0.12 tail:0.01 total:0.12 required:<0.50
[ultranest] done iterating.
Creating directory for new run ultranest_pairing/run1
[ultranest] Sampling 300 live points from prior ...


[ultranest] Explored until L=-8e+08  e+08 [-8.176e+08..-8.176e+08] | it/evals=14472/38078 eff=38.3080% N=300 
[ultranest] Likelihood function evaluations: 38078
[ultranest] Writing samples and results to disk ...
[ultranest] Writing samples and results to disk ... done
[ultranest]   logZ = -8.176e+08 +- 17.92
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -3404626148324666.00..-817589106.71 (KL: 14.43+-51.07 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 298 minimum live points (dlogz from 14.59 to 65.01, need <0.5)
[ultranest]   logZ error budget: single: 0.58 bs:17.92 tail:0.69 total:17.93 required:<0.50
[ultranest] Widening from 1 to 600 live points before L=-3e+15...
[ultranest] parent value is -inf, so widening roots
[ultranest] Widening roots to 600 live points (have 300 already) ...
[ultranest] Sampling 300 live points from prior ...


/home/luca/code/modellizazione dati/masse_nucleari/.venv/lib/python3.12/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Exploring (in particular: L=-inf..-817589106.71) ...
[ultranest] Explored until L=-8e+08  e+08 [-inf..-8.176e+08] | it/evals=29160/80794 eff=34.6261% N=600 
[ultranest] Likelihood function evaluations: 80863
[ultranest] Writing samples and results to disk ...
[ultranest] Writing samples and results to disk ... done
[ultranest]   logZ = -8.176e+08 +- 3.718
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -3404626148324666.00..-817589065.13 (KL: 2.83+-6.21 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 598 minimum live points (dlogz from 2.93 to 9.07, need <0.5)
[ultranest]   logZ error budget: single: 0.30 bs:3.72 tail:0.69 total:3.78 required:<0.50
[ultranest] Widening roots to 598 live points (have 600 already) ...
[ultranest] Exploring (in particular: L=-3404626148324666.00..-817589065.13) ...


...

/home/luca/code/modellizazione dati/masse_nucleari/.venv/lib/python3.12/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Explored until L=-8e+08  e+08 [-3.405e+15..-8.176e+08] | it/evals=59213/162176 eff=0.0000% N=1200 
[ultranest] Likelihood function evaluations: 162176
[ultranest] Writing samples and results to disk ...
[ultranest] Writing samples and results to disk ... done
[ultranest]   logZ = -8.176e+08 +- 40.6
[ultranest] Effective samples strategy satisfied (ESS = 1.0, need >400)
[ultranest] Posterior uncertainty strategy wants to improve: -3404626148324666.00..-817588784.83 (KL: 21.33+-145.71 nat, need <0.50 nat)
[ultranest] Evidency uncertainty strategy wants 1198 minimum live points (dlogz from 21.52 to 166.57, need <0.5)
[ultranest]   logZ error budget: single: 0.32 bs:40.60 tail:0.69 total:40.61 required:<0.50


/home/luca/code/modellizazione dati/masse_nucleari/.venv/lib/python3.12/site-packages/ultranest/integrator.py:1686: RuntimeWarning: invalid value encountered in divide
  p /= p.sum(axis=0).reshape((1, -1))


[ultranest] Widening from 1 to 2400 live points before L=-3e+15...
[ultranest] parent value is -inf, so widening roots
[ultranest] Widening roots to 2400 live points (have 1200 already) ...
[ultranest] Sampling 1200 live points from prior ...
[ultranest] Exploring (in particular: L=-inf..-817588784.83) ...


KeyboardInterrupt: 